# OpenVLA Attention Map Visualizer
**PoC: Attention-Guided Safety Filter for VLA** (arXiv:2606.09749)

**外科AI応用:** 術野内の非ターゲット組織（血管・神経）へのattentionを検出 → 行動ブロック

---
**実行方法:**
1. ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選択
2. 上から順にセルを実行（`Ctrl+F9` で全実行）
3. `MODE = 'synthetic'` で即動作、`MODE = 'real'` でOpenVLA本体を使用

In [ ]:
# ── 0. Config ─────────────────────────────────────────────────────
MODE        = 'synthetic'   # 'synthetic' | 'real'
INSTRUCTION = 'pick up the red cube'
IMAGE_PATH  = None          # 'real'モード時に画像パスを指定 e.g. '/content/scene.jpg'
THETA       = 0.50          # 衝突リスク閾値（これを超えたら UNSAFE）
TASK_BBOX   = None          # タスク対象BBox: (x0,y0,x1,y1) or None=中央50%を自動設定
N_LAYERS    = 32            # syntheticモード: シミュレートする層数
MODEL_ID    = 'openvla/openvla-7b'  # realモード用

In [ ]:
# ── 1. Install ────────────────────────────────────────────────────
if MODE == 'real':
    !pip install -q transformers accelerate timm einops
else:
    print('Synthetic mode: no extra installs needed')

import subprocess, sys
for pkg in ['matplotlib', 'numpy', 'Pillow']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('✓ Ready')

In [ ]:
# ── 2. Imports & Utilities ────────────────────────────────────────
import json, os, warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from PIL import Image
from IPython.display import display

warnings.filterwarnings('ignore')
OUT = Path('/content/attn_out')
OUT.mkdir(exist_ok=True)


def overlay_heatmap(base, attn, alpha=0.55, cmap='hot'):
    h, w = base.shape[:2]
    a = (attn - attn.min()) / (attn.max() - attn.min() + 1e-8)
    heat = (cm.get_cmap(cmap)(a)[..., :3] * 255).astype(np.uint8)
    heat = np.array(Image.fromarray(heat).resize((w, h)))
    return (base * (1 - alpha) + heat * alpha).astype(np.uint8)


def compute_risk(attn_grid, task_bbox, img_size):
    gh, gw = attn_grid.shape
    if task_bbox is None:
        cx, cy = gw // 2, gh // 2
        r = max(1, min(gh, gw) // 4)
        mask = np.zeros((gh, gw), dtype=bool)
        mask[cy-r:cy+r, cx-r:cx+r] = True
    else:
        x0, y0, x1, y1 = task_bbox
        sx, sy = gw / img_size, gh / img_size
        mask = np.zeros((gh, gw), dtype=bool)
        mask[int(y0*sy):int(y1*sy), int(x0*sx):int(x1*sx)] = True
    total = attn_grid.sum() + 1e-8
    tm = attn_grid[mask].sum() / total
    ntm = attn_grid[~mask].sum() / total
    return {'task_mass': float(tm), 'non_task_mass': float(ntm),
            'risk_score': float(ntm), 'safe': float(ntm) < THETA}

print('✓ Utilities loaded')

In [ ]:
# ── 3a. Synthetic data & scene ─────────────────────────────────────
def make_synthetic_scene(size=224):
    img = np.ones((size, size, 3), dtype=np.uint8) * 200
    img[size//2:] = [150, 120, 90]                                        # table
    cx, cy, s = int(size*.35), int(size*.55), size//10
    img[cy-s:cy+s, cx-s:cx+s] = [220, 50, 50]                            # red cube (task)
    dx, dy, r = int(size*.72), int(size*.30), size//14
    Y, X = np.ogrid[:size, :size]
    img[(X-dx)**2+(Y-dy)**2 <= r**2] = [50, 100, 220]                     # blue cylinder (distractor)
    img[:int(size*.45), int(size*.30):int(size*.42)] = [60, 60, 60]       # robot arm
    return img


def generate_synthetic_attention(n_layers=32, grid=16, seed=42):
    rng = np.random.default_rng(seed)
    x = np.linspace(0,1,grid); y = np.linspace(0,1,grid)
    xx, yy = np.meshgrid(x, y)
    layers = {}
    for li in range(n_layers):
        # Task object attention (center-left)
        a_task = np.exp(-((xx-.35)**2+(yy-.55)**2)/(2*(.12+.04*rng.random())**2))
        # Distractor attention grows in later layers
        dw = 0.10 + 0.25*(li/n_layers)
        a_dist = np.exp(-((xx-.72)**2+(yy-.30)**2)/(2*(.09+.03*rng.random())**2))
        attn = a_task + dw*a_dist + rng.exponential(.02,(grid,grid))
        attn /= attn.sum() + 1e-8
        layers[li] = attn
    return layers


if MODE == 'synthetic':
    scene = make_synthetic_scene()
    layers_attn = generate_synthetic_attention(N_LAYERS)
    grid_h = grid_w = 16
    Image.fromarray(scene).save(OUT/'scene.png')
    print(f'✓ Synthetic scene + {N_LAYERS} layers generated')
    plt.figure(figsize=(4,4))
    plt.imshow(scene); plt.title('Synthetic scene'); plt.axis('off'); plt.show()

In [ ]:
# ── 3b. Real OpenVLA inference (skip if synthetic) ────────────────
if MODE == 'real':
    import torch
    from transformers import AutoModelForVision2Seq, AutoProcessor

    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Device: {DEVICE}')

    print(f'Loading {MODEL_ID} …  (初回は数分かかります)')
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True, trust_remote_code=True
    ).to(DEVICE)
    model.eval()

    # Vision grid size
    vcfg = model.config.vision_config
    patch_size = getattr(vcfg, 'patch_size', 14)
    img_size   = getattr(vcfg, 'image_size', 224)
    grid_h = grid_w = img_size // patch_size
    N_IMG_TOK = grid_h * grid_w
    print(f'Vision grid: {grid_h}×{grid_w} = {N_IMG_TOK} tokens')

    # Hook setup
    captured = {}
    hooks = []
    for li, layer in enumerate(model.language_model.model.layers):
        def make_hook(idx):
            def hook(module, inp, out):
                if isinstance(out, tuple) and len(out)>1 and out[1] is not None:
                    w = out[1]                              # (B,H,T,T)
                    with torch.no_grad():
                        img_attn = w[0,:,-7:, 1:N_IMG_TOK+1]  # (H,7,N_img)
                        captured[idx] = img_attn.mean((0,1)).cpu().float().numpy()
            return hook
        hooks.append(layer.self_attn.register_forward_hook(make_hook(li)))

    # Inference
    image = Image.open(IMAGE_PATH).convert('RGB') if IMAGE_PATH else Image.new('RGB',(224,224),'grey')
    inputs = processor(INSTRUCTION, image).to(DEVICE, dtype=torch.bfloat16)
    with torch.no_grad():
        _ = model(**inputs, output_attentions=True)
    for h in hooks: h.remove()

    layers_attn = {li: v.reshape(grid_h, grid_w) for li, v in captured.items()}
    N_LAYERS = len(layers_attn)
    scene = np.array(image.resize((224,224)))
    print(f'✓ Inference done. Captured {N_LAYERS} layers.')

In [ ]:
# ── 4. Compute risk scores ─────────────────────────────────────────
IMG_SIZE = scene.shape[0]
risks = {}
for li, attn in layers_attn.items():
    risks[li] = compute_risk(attn, TASK_BBOX, IMG_SIZE)

all_risk_scores = [risks[i]['risk_score'] for i in range(N_LAYERS)]
agg_attn = np.mean(list(layers_attn.values()), axis=0)
agg_risk = compute_risk(agg_attn, TASK_BBOX, IMG_SIZE)

print(f"Aggregate risk  : {agg_risk['risk_score']:.3f}  (θ={THETA})")
print(f"Task mass       : {agg_risk['task_mass']:.3f}")
print(f"Non-task mass   : {agg_risk['non_task_mass']:.3f}")
print(f"Unsafe layers   : {sum(r['risk_score']>=THETA for r in risks.values())} / {N_LAYERS}")
print(f"VERDICT         : {'⚠ UNSAFE — BLOCK ACTION' if not agg_risk['safe'] else '✓ SAFE — EXECUTE'}")

In [ ]:
# ── 5. Aggregate visualization ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(scene)
axes[0].set_title('Original Scene', fontsize=12)
axes[0].axis('off')

blended = overlay_heatmap(scene, agg_attn)
axes[1].imshow(blended)
axes[1].set_title('Aggregate Attention\n(action token → image patches)', fontsize=12)
axes[1].axis('off')

colors = ['red' if s >= THETA else 'steelblue' for s in all_risk_scores]
axes[2].bar(range(N_LAYERS), all_risk_scores, color=colors, width=0.8)
axes[2].axhline(THETA, color='orange', ls='--', lw=1.5, label=f'θ={THETA}')
axes[2].set_xlabel('Layer'); axes[2].set_ylabel('Collision Risk Score')
axes[2].set_title('Risk Score per Layer', fontsize=12)
axes[2].set_ylim(0, 1); axes[2].legend()

verdict = '⚠ UNSAFE — ACTION BLOCKED' if not agg_risk['safe'] else '✓ SAFE — ACTION OK'
color   = 'red' if not agg_risk['safe'] else 'darkgreen'
fig.suptitle(f"{verdict}  |  risk={agg_risk['risk_score']:.3f}",
             fontsize=13, fontweight='bold', color=color)
fig.tight_layout()
fig.savefig(OUT/'attention_aggregate.png', dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved: {OUT}/attention_aggregate.png')

In [ ]:
# ── 6. Per-layer grid ──────────────────────────────────────────────
SHOW_LAYERS = min(N_LAYERS, 32)  # 多い場合は先頭32層のみ表示
n_cols = 8
n_rows = (SHOW_LAYERS + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*2.5, n_rows*2.5))
axes = axes.flatten()

for li in range(SHOW_LAYERS):
    blended = overlay_heatmap(scene, layers_attn[li])
    r = risks[li]
    axes[li].imshow(blended)
    c = 'red' if r['risk_score'] >= THETA else 'green'
    axes[li].set_title(f"L{li:02d} {r['risk_score']:.2f}", fontsize=7, color=c)
    axes[li].axis('off')

for i in range(SHOW_LAYERS, len(axes)):
    axes[i].axis('off')

fig.suptitle('Action Token → Image Patch Attention (all layers)', fontsize=11)
fig.tight_layout()
fig.savefig(OUT/'attention_all_layers.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Saved: {OUT}/attention_all_layers.png')

In [ ]:
# ── 7. θ感度分析 ────────────────────────────────────────────────────
thetas = np.linspace(0.1, 0.9, 17)
n_blocked = [sum(r['risk_score'] >= t for r in risks.values()) for t in thetas]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thetas, n_blocked, 'o-', color='steelblue', linewidth=2)
ax.axvline(THETA, color='orange', ls='--', lw=1.5, label=f'Current θ={THETA}')
ax.axvline(agg_risk['risk_score'], color='red', ls=':', lw=1.5,
           label=f'Agg risk={agg_risk["risk_score"]:.2f}')
ax.fill_between(thetas, n_blocked, alpha=0.15, color='steelblue')
ax.set_xlabel('Threshold θ', fontsize=11)
ax.set_ylabel('Unsafe layers (blocked)', fontsize=11)
ax.set_title('θ Sensitivity — 閾値を下げると敏感になる', fontsize=12)
ax.legend(fontsize=10)
ax.set_ylim(0, N_LAYERS)
fig.tight_layout()
fig.savefig(OUT/'theta_sensitivity.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8. JSON report & download ──────────────────────────────────────
report = {
    'mode': MODE,
    'instruction': INSTRUCTION,
    'theta': THETA,
    'verdict': 'UNSAFE' if not agg_risk['safe'] else 'SAFE',
    'aggregate': agg_risk,
    'n_unsafe_layers': sum(r['risk_score'] >= THETA for r in risks.values()),
    'n_layers': N_LAYERS,
    'per_layer': [
        {'layer': li, **risks[li]} for li in range(N_LAYERS)
    ]
}
with open(OUT/'attention_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print('=== FINAL REPORT ===')
print(f"Verdict        : {report['verdict']}")
print(f"Risk score     : {agg_risk['risk_score']:.3f}  (θ={THETA})")
print(f"Task mass      : {agg_risk['task_mass']:.3f}")
print(f"Non-task mass  : {agg_risk['non_task_mass']:.3f}")
print(f"Unsafe layers  : {report['n_unsafe_layers']} / {N_LAYERS}")
print(f"\nFiles saved in: {OUT}")
for p in sorted(OUT.iterdir()):
    print(f"  {p.name}")

# ── 環境に応じてダウンロード方法を切り替え ──────────────────────────────
import os, sys
IN_BROWSER_COLAB = 'COLAB_BACKEND_VERSION' in os.environ and 'google.colab' in sys.modules
OUTPUT_FILES = ['attention_aggregate.png', 'attention_all_layers.png',
                'theta_sensitivity.png', 'attention_report.json']

if IN_BROWSER_COLAB:
    # ブラウザColab: files.download() でローカルへ
    from google.colab import files
    for fname in OUTPUT_FILES:
        p = OUT / fname
        if p.exists():
            files.download(str(p))
            print(f'Downloaded: {fname}')
else:
    # colab exec (CLI) モード — VM上に保存済み
    # ローカルに取得するには colab download を使う
    print('\n[CLI mode] ファイルはVM上の /content/attn_out/ に保存されました。')
    print('ローカルへの取得コマンド:')
    for fname in OUTPUT_FILES:
        print(f'  colab download /content/attn_out/{fname} ./{fname}')

## 次のステップ

| ステップ | 内容 |
|---|---|
| **A. 実画像で試す** | `IMAGE_PATH = '/content/scene.jpg'` にアップロード → `MODE='real'` |
| **B. θをキャリブレーション** | セル7のθ感度グラフを見て適切な閾値を決定 |
| **C. task_bbox を設定** | `TASK_BBOX = (x0,y0,x1,y1)` でターゲット物体を明示 |
| **D. 外科シーン** | 腹腔鏡動画フレームを入力 → 非ターゲット組織へのattentionを検出 |
| **E. ProbeActへ** | 失敗予測Linear Probe（arXiv:2606.09740）へ進む |

---
*PRJ-017 AI Task Platform — Safety Filter for VLA PoC*